# تشغيل GL-log-MoRA على Qwen2.5-3B-Instruct

هذا الدفتر **مشغّل Cloud عام فقط**. لا يحتوي على تنفيذ GL-log-MoRA؛ بل يستنسخ مستودع `qwen25-3b-yemeni-mora`، ويستنسخ مستودع MoRA الرسمي، ثم يطبّق patch محفوظًا داخل المستودع قبل تثبيت الكود وتشغيل التدريب والاختبار والدردشة.

الكود الفعلي والتعديلات والاختبارات موجودة في مستودع المشروع. يمكن تشغيل الدفتر في أي بيئة Cloud/Jupyter تدعم Python وGPU، وليس مرتبطًا بواجهات Google Colab الخاصة.

In [ ]:
# 1) فحص الموارد والبيئة
import os, platform, shutil, subprocess, sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi غير متاح؛ سيعمل التدريب على CPU إن كانت الذاكرة كافية.")


## 2. استنساخ مشروع التشغيل وMoRA الرسمي

يُستنسخ مشروع `qwen25-3b-yemeni-mora` بوصفه مشروع التشغيل، ثم يُستنسخ `kongds/MoRA` من المصدر الرسمي. قبل التثبيت، يشغّل الدفتر فحصًا على MoRA الأصلي، ثم يطبّق patch GL-log-MoRA الموجود في المشروع. تحفظ معلومات المصدر والـcommit قبل التعديل.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

WORKDIR = Path(os.environ.get("GL_LOG_MORA_WORKDIR", str(Path.cwd() / "gl_log_mora_run"))).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
RUNNER_REPO_URL = "https://github.com/EngKHALIDx/qwen25-3b-yemeni-mora.git"
RUNNER_BRANCH = os.environ.get("GL_LOG_MORA_BRANCH", "moranew")
RUNNER_DIR = WORKDIR / "qwen25-3b-yemeni-mora"
MORA_REPO_URL = "https://github.com/kongds/MoRA.git"
MORA_DIR = WORKDIR / "MoRA"
PATCH_FILE = RUNNER_DIR / "patches" / "gl_log_mora_mora.patch"

def run(command, cwd=None):
    print("$", " ".join(map(str, command)))
    return subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)

def clone_or_update(url, directory, branch=None):
    if not (directory / ".git").exists():
        command = ["git", "clone", "--depth", "1"]
        if branch:
            command += ["--branch", branch, "--single-branch"]
        command += [url, str(directory)]
        run(command)
    else:
        run(["git", "fetch", "--all", "--prune"], cwd=directory)

clone_or_update(RUNNER_REPO_URL, RUNNER_DIR, RUNNER_BRANCH)
clone_or_update(MORA_REPO_URL, MORA_DIR)
assert (RUNNER_DIR / "train.py").exists(), "لم يُعثر على train.py في مشروع التشغيل"
assert PATCH_FILE.exists(), f"ملف patch غير موجود: {PATCH_FILE}"

os.chdir(RUNNER_DIR)
base_mora_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=MORA_DIR, text=True).strip()
run([sys.executable, "-m", "compileall", "-q", str(MORA_DIR / "peft-mora" / "src")])
marker = MORA_DIR / ".gl_log_mora_applied"
if not marker.exists():
    run(["git", "apply", "--check", str(PATCH_FILE)], cwd=MORA_DIR)
    run(["git", "apply", str(PATCH_FILE)], cwd=MORA_DIR)
    marker.write_text("GL-log-MoRA patch applied\n", encoding="utf-8")
else:
    print("GL-log-MoRA patch already applied.")
run([sys.executable, "-m", "compileall", "-q", str(MORA_DIR / "peft-mora" / "src")])

patch_sha256 = hashlib.sha256(PATCH_FILE.read_bytes()).hexdigest()
provenance = {
    "runner_repository": RUNNER_REPO_URL,
    "runner_branch": RUNNER_BRANCH,
    "mora_repository": MORA_REPO_URL,
    "mora_commit_before_patch": base_mora_commit,
    "gl_log_mora_patch": str(PATCH_FILE),
    "gl_log_mora_patch_sha256": patch_sha256,
    "workdir": str(WORKDIR),
}
(WORKDIR / "provenance.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(provenance, ensure_ascii=False, indent=2))


In [ ]:
# 3) تثبيت اعتماديات مشروع التشغيل وPEFT MoRA الرسمي بعد تطبيق patch
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(RUNNER_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(MORA_DIR / "peft-mora")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
print("Installed patched PEFT from:", MORA_DIR / "peft-mora")


## 4. مسار المخرجات

يستخدم الدفتر مجلدًا محليًا داخل `GL_LOG_MORA_WORKDIR` أو المجلد الحالي. لا يعتمد على Google Drive أو واجهات Colab؛ يمكن نسخ المخرجات إلى تخزين Cloud خارجي وفق بيئة التشغيل.

In [ ]:
from pathlib import Path

OUTPUT_DIR = WORKDIR / "outputs" / "qwen25_3b_mora_adapter"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR)


## 5. تنزيل ملف البيانات من الرابط المعتمد

يُنزّل الملف من Google Drive باستخدام `gdown`، وهي أداة تعمل في بيئات Cloud عامة ولا تتطلب واجهة Colab. بعد التنزيل يتحقق الدفتر من SHA-256 ثم ينشئ ملف JSONL موحّدًا للتدريب.

In [ ]:
import hashlib, json, subprocess, sys
from pathlib import Path

DRIVE_FILE_ID = "1U9-DSU0_GH4LXu1SmmPeKoK7dhxwrFPz"
EXPECTED_CONTENT_SHA256 = "bbd4721b20b02c2d5a97b594c530354548a2dcc01b5c8f508539abe247400dfe"
DATA_DIR = WORKDIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_DATA_PATH = DATA_DIR / "qwen25_3b_semantic_colab.jsonl"
DATA_PATH = DATA_DIR / "qwen25_3b_semantic_colab_train.jsonl"

if not SOURCE_DATA_PATH.exists() or SOURCE_DATA_PATH.stat().st_size < 100_000_000:
    subprocess.run([
        sys.executable, "-m", "gdown", "--id", DRIVE_FILE_ID,
        "-O", str(SOURCE_DATA_PATH), "--fuzzy"
    ], check=True)
print("Source data path:", SOURCE_DATA_PATH)
print("Source bytes:", SOURCE_DATA_PATH.stat().st_size)


In [ ]:
# 5.1) توحيد مخطط البيانات قبل استخدامه في التدريب
import json

if not DATA_PATH.exists() or DATA_PATH.stat().st_size < 100_000_000:
    normalized_count = 0
    with SOURCE_DATA_PATH.open("r", encoding="utf-8") as src, DATA_PATH.open("w", encoding="utf-8") as dst:
        for raw_line in src:
            if not raw_line.strip():
                continue
            record = json.loads(raw_line)
            messages = record.get("messages")
            if not isinstance(messages, list) or not messages:
                raise ValueError(f"Invalid messages at normalized row {normalized_count + 1}")
            metadata = {key: value for key, value in record.items() if key not in {"messages", "metadata_json"}}
            if "metadata_json" in record:
                metadata["metadata_json_original"] = record["metadata_json"]
            normalized = {
                "messages": messages,
                "metadata_json": json.dumps(metadata, ensure_ascii=False, separators=(",", ":")),
            }
            dst.write(json.dumps(normalized, ensure_ascii=False, separators=(",", ":")) + "\n")
            normalized_count += 1
    print("Normalized records:", normalized_count)
else:
    print("Normalized file already exists:", DATA_PATH)
print("Training data path:", DATA_PATH)


In [ ]:
# 6) تحقق من المصدر والبنية والعدد بعد التوحيد
import hashlib, json

source_digest = hashlib.sha256()
with SOURCE_DATA_PATH.open("rb") as raw:
    for block in iter(lambda: raw.read(1024 * 1024), b""):
        source_digest.update(block)
assert source_digest.hexdigest() == EXPECTED_CONTENT_SHA256, source_digest.hexdigest()

count = 0
malformed = 0
no_assistant = 0
max_token_count = 0
with DATA_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        try:
            record = json.loads(line)
            messages = record.get("messages")
            roles = {m.get("role") for m in messages if isinstance(m, dict)} if isinstance(messages, list) else set()
            if not isinstance(messages, list) or roles != {"system", "user", "assistant"}:
                malformed += 1
            if not any(isinstance(m, dict) and m.get("role") == "assistant" and m.get("content", "").strip() for m in messages if isinstance(m, dict)):
                no_assistant += 1
            metadata = json.loads(record.get("metadata_json", "{}"))
            max_token_count = max(max_token_count, int(metadata.get("token_count_qwen25_chat", 0)))
            count += 1
        except (json.JSONDecodeError, TypeError, ValueError):
            malformed += 1

assert count == 165303, count
assert malformed == 0, malformed
assert no_assistant == 0, no_assistant
print({"status": "PASS", "records": count, "max_stored_token_count": max_token_count, "source_sha256": source_digest.hexdigest(), "normalized_path": str(DATA_PATH)})


## 7. اختبار دخان حقيقي

لا يبدأ التدريب الكامل قبل نجاح هذه الخلية. الاختبار يشغّل `train.py` الموجود في المستودع مع MoRA الرسمي بعد تطبيق patch GL-log-MoRA، ثم يمرر سجلات حقيقية من البيانات.

In [ ]:
import subprocess, sys

def run_training(extra_args):
    command = [sys.executable, "train.py", "--config", "configs/qwen25_3b_mora.json", "--data_path", str(DATA_PATH)] + list(extra_args)
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=str(RUNNER_DIR), check=True)

SMOKE_OUTPUT_DIR = OUTPUT_DIR.parent / "smoke_test"
run_training(["--max_train_samples", "1", "--max_steps", "1", "--output_dir", str(SMOKE_OUTPUT_DIR)])


## 8. التدريب الكامل

بعد نجاح اختبار الدخان، شغّل هذه الخلية. الإعدادات تُقرأ من `configs/qwen25_3b_mora.json` في المستودع، ويُحفظ الـadapter في مجلد المخرجات.

In [ ]:
run_training(["--output_dir", str(OUTPUT_DIR)])


## 9. استئناف التدريب من checkpoint

غيّر مسار checkpoint ثم شغّل الخلية عند الحاجة. لا يُحذف أي checkpoint تلقائيًا إلا وفق `save_total_limit` الموجود في إعدادات التدريب.

In [ ]:
# مثال، لا تشغّلها إلا بعد تعديل المسار:
# run_training([
#     "--output_dir", str(OUTPUT_DIR),
#     "--resume_from_checkpoint", str(OUTPUT_DIR / "checkpoint-500"),
# ])


## 10. اختبار الـadapter بعد التدريب

In [ ]:
import json, sys, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# استيراد PEFT من MoRA الرسمي بعد تطبيق patch، لا من نسخة النظام.
sys.path.insert(0, str(MORA_DIR / "peft-mora" / "src"))
import peft
from peft import LoraConfig, PeftModel
assert "use_mora" in getattr(LoraConfig, "__dataclass_fields__", {}), (peft.__file__, "MoRA field missing")
print("PEFT path:", peft.__file__)

adapter_dir = str(OUTPUT_DIR)
base_id = "Qwen/Qwen2.5-3B-Instruct"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
base = AutoModelForCausalLM.from_pretrained(
    base_id, quantization_config=bnb, torch_dtype=torch.float16, device_map="auto"
)
model = PeftModel.from_pretrained(base, adapter_dir)
model.eval()

test_record = None
with DATA_PATH.open("r", encoding="utf-8") as handle:
    for line in handle:
        record = json.loads(line)
        metadata = json.loads(record.get("metadata_json", "{}"))
        if record.get("task_type", metadata.get("task_type")) == "rule":
            test_record = record
            break
assert test_record is not None, "لم يُعثر على سجل rule"
test_messages = test_record["messages"][:2]
inputs = tokenizer.apply_chat_template(test_messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(answer)


## 10.1. دردشة مباشرة مع الـadapter

عدّل `USER_QUESTION` ثم شغّل الخلية. تستخدم الخلية النموذج والـadapter المحفوظين في المخرجات.

In [ ]:
USER_QUESTION = "هل تصح شهادة شخص سمع الزوج يعترف بإطلاق النار على زوجته إذا أنكر الزوج لاحقًا؟"
MAX_NEW_TOKENS = 256

assert "model" in globals() and "tokenizer" in globals(), "شغّل خلية اختبار adapter أولاً."
chat_messages = [
    {"role": "system", "content": "أجب باللغة العربية بوضوح، واذكر حدود إجابتك عند الحاجة."},
    {"role": "user", "content": USER_QUESTION},
]
chat_inputs = tokenizer.apply_chat_template(chat_messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
with torch.no_grad():
    chat_outputs = model.generate(chat_inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id)
chat_answer = tokenizer.decode(chat_outputs[0][chat_inputs.shape[-1]:], skip_special_tokens=True)
print(chat_answer.strip())


In [ ]:
# 11) حفظ سجل مختصر للتشغيل
from datetime import datetime, timezone
run_log = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "data_sha256": EXPECTED_CONTENT_SHA256,
    "records": 165303,
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "mora_source": "https://github.com/kongds/MoRA",
    "mora_commit_before_patch": provenance["mora_commit_before_patch"],
    "gl_log_mora": True,
    "gl_log_lambda": 0.01,
    "gl_log_delta": 0.001,
    "gl_log_q_lr": 0.0003,
    "mora_type": 6,
    "output_dir": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "cloud_run_log.json").write_text(json.dumps(run_log, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(run_log, ensure_ascii=False, indent=2))


## 12. ضغط الـadapter وتنزيله من بيئة Cloud

بعد اكتمال التدريب، شغّل هذه الخلية لإنشاء ملف ZIP. طريقة تنزيل الملف تعتمد على منصة Cloud المستخدمة.

In [ ]:
import shutil
from pathlib import Path

if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
    print("Adapter archive:", archive)
    try:
        from IPython.display import FileLink, display
        display(FileLink(archive))
    except Exception:
        pass
else:
    print("لم يُعثر على مخرجات تدريب لضغطها.")
